# Prototype: Erste Exploration der Rohdaten

Dieses Notebook dient als erster Kontakt mit den NYC Parking Violations Rohdaten. Ziel ist es, ein grundlegendes Verständnis der Daten zu gewinnen, bevor das Pre-processing durchgeführt wird.

## Ziel

- Rohdaten aus HDFS laden
- 1%-Sample ziehen für schnelle Exploration ohne lange Wartezeiten
- Struktur des Datensatzes verstehen
- Erste Zeilen anschauen
- Null-Werte und fehlende Felder identifizieren
- Eindeutigkeit des Primary Keys prüfen
- Verteilung pro Fiskaljahr prüfen

Die Erkenntnisse aus diesem Notebook fliessen ins Pre-processing (`src/1_Pre_Processing/5.0_clean_parking_violations.ipynb`) ein.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, sum as spark_sum

spark = SparkSession.builder \
    .appName("BDLC_Parking_Violations_RawPrototype") \
    .master("spark://bdlc-012.bdlc.ls.eee.intern:7077") \
    .config("spark.executor.cores", "4") \
    .config("spark.executor.memory", "15g") \
    .config("spark.cores.max", "12") \
    .getOrCreate()

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/30 11:26:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/30 11:26:07 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
# Rohdaten laden
raw_paths = {
    2023: "hdfs:///parking_violations/raw/2023/parking_violations_2023.csv",
    2024: "hdfs:///parking_violations/raw/2024/parking_violations_2024.csv",
    2025: "hdfs:///parking_violations/raw/2025/parking_violations_2025.csv",
}

dfs = []
for fiscal_year, path in raw_paths.items():
    df_year = spark.read.csv(path, header=True, inferSchema=False) \
        .withColumn("Fiscal Year", lit(fiscal_year))
    dfs.append(df_year)

df_raw = dfs[0]
for df_next in dfs[1:]:
    df_raw = df_raw.unionByName(df_next)

print(f"Zeilen: {df_raw.count():,}")
print(f"Spalten: {len(df_raw.columns)}")

[Stage 3:========================================================>(75 + 1) / 76]

Zeilen: 54,223,582
Spalten: 44


In [3]:
# 1%-Sample ziehen
sample_raw = df_raw.sample(fraction=0.01, seed=42)
print(f"Sample-Grösse: {sample_raw.count():,} Zeilen")

26/05/30 11:26:25 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
[Stage 6:======================================================>  (73 + 3) / 76]

Sample-Grösse: 542,412 Zeilen


In [4]:
# Schema anschauen
sample_raw.printSchema()

root
 |-- Summons Number: string (nullable = true)
 |-- Plate ID: string (nullable = true)
 |-- Registration State: string (nullable = true)
 |-- Plate Type: string (nullable = true)
 |-- Issue Date: string (nullable = true)
 |-- Violation Code: string (nullable = true)
 |-- Vehicle Body Type: string (nullable = true)
 |-- Vehicle Make: string (nullable = true)
 |-- Issuing Agency: string (nullable = true)
 |-- Street Code1: string (nullable = true)
 |-- Street Code2: string (nullable = true)
 |-- Street Code3: string (nullable = true)
 |-- Vehicle Expiration Date: string (nullable = true)
 |-- Violation Location: string (nullable = true)
 |-- Violation Precinct: string (nullable = true)
 |-- Issuer Precinct: string (nullable = true)
 |-- Issuer Code: string (nullable = true)
 |-- Issuer Command: string (nullable = true)
 |-- Issuer Squad: string (nullable = true)
 |-- Violation Time: string (nullable = true)
 |-- Time First Observed: string (nullable = true)
 |-- Violation County: str

In [5]:
# Erste Zeilen anschauen
sample_raw.show(5, truncate=False)

26/05/30 11:26:29 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------+--------+------------------+----------+----------+--------------+-----------------+------------+--------------+------------+------------+------------+-----------------------+------------------+------------------+---------------+-----------+--------------+------------+--------------+-------------------+----------------+---------------------------------+------------+-------------+-------------------+-------------------+-----------+------------+--------------------+----------------------+--------------------+------------------+-------------+---------------------+------------+------------+--------------+-------------------+---------------------+---------------------------------+-----------------+------------------------+-----------+
|Summons Number|Plate ID|Registration State|Plate Type|Issue Date|Violation Code|Vehicle Body Type|Vehicle Make|Issuing Agency|Street Code1|Street Code2|Street Code3|Vehicle Expiration Date|Violation Location|Violation Precinct|Issuer Precinct|I

In [6]:
# Null-Werte prüfen
sample_raw.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in sample_raw.columns
]).show(truncate=False)

[Stage 10:=======================================================>(75 + 1) / 76]

+--------------+--------+------------------+----------+----------+--------------+-----------------+------------+--------------+------------+------------+------------+-----------------------+------------------+------------------+---------------+-----------+--------------+------------+--------------+-------------------+----------------+---------------------------------+------------+-----------+-------------------+-------------------+-----------+------------+--------------------+----------------------+--------------------+------------------+-------------+---------------------+------------+------------+--------------+-------------------+---------------------+---------------------------------+-----------------+------------------------+-----------+
|Summons Number|Plate ID|Registration State|Plate Type|Issue Date|Violation Code|Vehicle Body Type|Vehicle Make|Issuing Agency|Street Code1|Street Code2|Street Code3|Vehicle Expiration Date|Violation Location|Violation Precinct|Issuer Precinct|Iss

In [7]:
# Eindeutigkeit des Primary Keys prüfen (Summons Number)
total = sample_raw.count()
distinct = sample_raw.select("Summons Number").distinct().count()

print(f"Gesamtzeilen:            {total:,}")
print(f"Distinct Summons Number: {distinct:,}")
print(f"Duplikate:               {total - distinct:,}")

[Stage 16:======================================================> (74 + 2) / 76]

Gesamtzeilen:            542,412
Distinct Summons Number: 542,009
Duplikate:               403


In [8]:
# Verteilung pro Fiskaljahr
sample_raw.groupBy("Fiscal Year").count().orderBy("Fiscal Year").show()

[Stage 22:=====================================================>  (73 + 3) / 76]

+-----------+------+
|Fiscal Year| count|
+-----------+------+
|       2023|216095|
|       2024|161132|
|       2025|165185|
+-----------+------+



## Erkenntnisse aus der Rohdaten-Exploration

Die Exploration des 1%-Samples der Rohdaten (542'412 Zeilen) zeigt folgende Auffälligkeiten:

1. **`Violation Description`:** 10'626 Einträge im Sample haben keine Beschreibung. Die vorhandenen Beschreibungen wirken inkonsistent — verschiedene Formulierungen für ähnliche Verstösse sind sichtbar. Eine einheitliche Beschreibung pro Code wäre für Analysen sinnvoll.

2. **`Violation Time` im ungewöhnlichen Format:** Das Zeitfeld enthält Werte wie `0834A` oder `0215P` — ein 12-Stunden-Format das speziell geparst werden muss. Vereinzelt gibt es ungültige Werte ohne A/P-Suffix.

3. **Datums-Tippfehler:** Die Jahresverteilung zeigt Einträge weit ausserhalb des erwarteten Bereichs (z.B. 2000, 2012, 2027, 2052). Diese sollten gefiltert werden.

4. **Duplikate:** Im Sample wurden 403 Duplikate auf `Summons Number` gefunden — der vermeintliche Primary Key ist nicht eindeutig. Eine Deduplizierung ist notwendig.

Diese Erkenntnisse fliessen direkt ins Pre-processing ein.

In [9]:
spark.stop()